# DWG — cat, FFN-only in layers 0–7

Eval-time DWG ablation driven by `configs/dwg_qwen_cat_ffn_early.yaml`:

* `full`      — baseline, LoRA active everywhere (hits the existing cache).
* `ffn_early` — LoRA kept ONLY on `{gate_proj, up_proj, down_proj}` in layers 0–7, zeroed everywhere else.

Pattern mirrors `dwg_results.ipynb` (coverage → rank curves → subspace share → per-seed scatter → headline table).

Registry is read from `$ARTIFACTS_DIR/registry.json`.

In [ ]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sl.config as sl_config

REGISTRY_PATH = Path(sl_config.ARTIFACTS_DIR) / "registry.json"
ANIMAL = "cat"
TARGET_MODE = "ffn_early"
MODE_COLORS = {"full": "#1f77b4", TARGET_MODE: "#ff7f0e", "reference": "#7f7f7f"}

print(f"Registry: {REGISTRY_PATH}")
with open(REGISTRY_PATH) as f:
    reg = json.load(f)
print(f"  experiments: {len(reg['experiments'])}")

## 1. Load & filter

* `df_ablation` — runs from this sweep: `svd_mode='full'` × `dwg_mode='ffn_early'`.
* `df_ref` — reference full-LoRA curve, assembled from any config whose effective transform is the identity
  (`dwg_mode ∈ {None, 'full'}` × `svd_mode ∈ {None, 'full'}`). Same dataset/system-prompt as this sweep.

In [ ]:
def _extract_results(e: dict) -> dict:
    cfg = e.get("config") or {}
    agg = ((e.get("results") or {}).get("aggregate") or {}).get("clean") or {}
    gen = ((e.get("results") or {}).get("generation_aggregate") or {}).get("clean") or {}
    return {
        "animal": cfg.get("target_animal"),
        "svd_mode": cfg.get("svd_mode"),
        "dwg_mode": cfg.get("dwg_mode"),
        "rank": cfg.get("lora_rank"),
        "gen_seed": cfg.get("generation_seed"),
        "train_seed": cfg.get("training_seed"),
        "system_prompt_variant": cfg.get("system_prompt_variant"),
        "generation_strategy": cfg.get("generation_strategy"),
        "status": e.get("status"),
        "updated_at": e.get("updated_at"),
        "mean_p": agg.get("mean_probability"),
        "median_p": agg.get("median_probability"),
        "mean_rank": agg.get("mean_rank"),
        "prob_ratio": agg.get("probability_ratio"),
        "mean_p_contains": gen.get("mean_p_contains"),
    }

rows = []
for key, e in reg["experiments"].items():
    cfg = e.get("config") or {}
    if cfg.get("target_animal") != ANIMAL:
        continue
    r = _extract_results(e)
    r["key"] = key
    rows.append(r)

df_all = pd.DataFrame(rows)
print(f"All cat runs: {len(df_all)}")

df_ablation = df_all[
    (df_all.svd_mode == "full")
    & (df_all.dwg_mode == TARGET_MODE)
    & (df_all.status == "completed")
].copy()

df_ref = df_all[
    (df_all.dwg_mode.isna() | (df_all.dwg_mode == "full"))
    & (df_all.svd_mode.isna() | (df_all.svd_mode == "full"))
    & (df_all.status == "completed")
    & (df_all.generation_strategy == "filtered")
    & (df_all.system_prompt_variant == "subliminal")
].copy()

print(f"ffn_early rows: {len(df_ablation)}")
print(f"Reference full-LoRA rows: {len(df_ref)}")

## 2. Coverage sanity check

For each rank the ablation sweep should have **9 seeds** (3 gen × 3 train).

In [ ]:
print("ffn_early coverage (rank \u2192 n):")
display(df_ablation.groupby("rank").size().to_frame("n").T)

print("Reference full-LoRA coverage (rank \u2192 n):")
display(df_ref.groupby("rank").size().to_frame("n").T)

## 3. Rank curves

`ffn_early` vs. the reference full-LoRA sweep. `mean_p` is the first-token probability for "cat"
(clean metric for cat, which tokenizes as a single token). `mean_p_contains` is robustness backup.

In [ ]:
def _rank_curve(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    grouped = df.groupby("rank")[metric].agg(["mean", "std", "count"]).reset_index()
    grouped["sem"] = grouped["std"] / np.sqrt(grouped["count"].clip(lower=1))
    return grouped.sort_values("rank")


def plot_rank_curves(metric: str, ylabel: str, log_y: bool = False):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    sub = df_ablation.dropna(subset=[metric])
    if not sub.empty:
        curve = _rank_curve(sub, metric)
        ax.errorbar(curve["rank"], curve["mean"], yerr=curve["sem"], marker="o",
                    color=MODE_COLORS[TARGET_MODE], label=f"DWG {TARGET_MODE}")
    ref = df_ref.dropna(subset=[metric])
    if not ref.empty:
        curve = _rank_curve(ref, metric)
        ax.errorbar(curve["rank"], curve["mean"], yerr=curve["sem"], marker="s",
                    color=MODE_COLORS["reference"], linestyle="--",
                    label="full LoRA (ref sweep)")
    ax.set_xscale("log", base=2)
    if log_y:
        ax.set_yscale("log")
    ax.set_xlabel("LoRA rank")
    ax.set_ylabel(ylabel)
    ax.set_title(f"cat — {ylabel}")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9, loc="best")
    fig.tight_layout()
    return fig

plot_rank_curves("mean_p", "mean P(first token = cat)");

In [ ]:
plot_rank_curves("mean_p_contains", "mean P(generation contains 'cat')");

## 4. Subspace share

`ffn_early / full_reference` per rank. > 1 means the FFN-early sub-adapter beats the full adapter at that rank.

In [ ]:
def share(metric: str) -> pd.DataFrame:
    abl_mean = df_ablation.groupby("rank")[metric].mean().reset_index()
    ref_mean = df_ref.groupby("rank")[metric].mean().reset_index().rename(columns={metric: "ref"})
    merged = abl_mean.merge(ref_mean, on="rank", how="left")
    merged["share"] = merged[metric] / merged["ref"]
    return merged

def plot_share(metric: str, title: str):
    s = share(metric).dropna(subset=["share"])
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(s["rank"], s["share"], marker="o", color=MODE_COLORS[TARGET_MODE], label=TARGET_MODE)
    ax.axhline(1.0, color="black", linestyle=":", alpha=0.6, label="full LoRA")
    ax.axhline(0.0, color="black", linestyle="-", alpha=0.3)
    ax.set_xscale("log", base=2)
    ax.set_xlabel("LoRA rank")
    ax.set_ylabel(f"{metric} / reference full-LoRA")
    ax.set_title(f"cat — {title}")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    fig.tight_layout()
    return fig

plot_share("mean_p", "P(first token = cat)");
plot_share("mean_p_contains", "P(generation contains 'cat')");

## 5. Per-seed scatter

Confirms the rank-curve is not driven by one unusual (gen_seed, train_seed) cell.

In [ ]:
def plot_seed_scatter(metric: str, title: str):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for label, df, color in [
        ("full LoRA (ref)", df_ref, MODE_COLORS["reference"]),
        (TARGET_MODE, df_ablation, MODE_COLORS[TARGET_MODE]),
    ]:
        sub = df.dropna(subset=[metric])
        if sub.empty:
            continue
        rng = np.random.RandomState(hash(label) & 0xFFFF)
        jitter = rng.uniform(0.90, 1.10, size=len(sub))
        ax.scatter(sub["rank"] * jitter, sub[metric], color=color, alpha=0.55, s=32,
                   label=label, edgecolor="white", linewidth=0.5)
        means = sub.groupby("rank")[metric].mean()
        ax.plot(means.index, means.values, color=color, marker="o", alpha=0.9)
    ax.set_xscale("log", base=2)
    ax.set_xlabel("LoRA rank")
    ax.set_ylabel(metric)
    ax.set_title(f"cat per-seed scatter — {title}")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    fig.tight_layout()
    return fig

plot_seed_scatter("mean_p", "P(first token = cat)");
plot_seed_scatter("mean_p_contains", "P(generation contains 'cat')");

## 6. Headline table

Rank-8 summary: compare `ffn_early` to the reference full-LoRA at matched rank.

In [ ]:
def headline(rank: int = 8) -> pd.DataFrame:
    rows = []
    for label, df in [("full (ref)", df_ref), (TARGET_MODE, df_ablation)]:
        sub = df[df["rank"] == rank]
        rows.append({
            "variant": label,
            "n": len(sub.dropna(subset=["mean_p"])),
            "mean_p": sub["mean_p"].mean(),
            "mean_p_contains": sub["mean_p_contains"].mean(),
            "mean_rank": sub["mean_rank"].mean(),
        })
    return pd.DataFrame(rows)

headline(rank=8).round(4)

## Notes

Before running these cells:

```bash
./submit.sh benchmark-parallel --config configs/dwg_qwen_cat_ffn_early.yaml --array-size 162 --max-gpus 6
```

Only the `ffn_early` variant actually runs on GPU (~81 new evals); `full` hits the existing cache
from the main cat_filtered / dwg_qwen_cat sweeps, so the ref curve is already populated.